In [ ]:
%pip install neo4j datasets google-genai pydantic python-dotenv numpy tqdm --quiet

In [ ]:
import os
import pickle
import sys
from pathlib import Path

from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv("../.env")
sys.path.insert(0, str(Path("..").resolve()))

In [ ]:
from qasa_rag import KnowledgeGraphLoader

NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

In [ ]:
DATASET_NAME = "framolfese/2WikiMultihopQA"
MAX_QUESTIONS = 1000

dataset_dict = load_dataset(DATASET_NAME, split="validation")

In [ ]:
raw_dataset = dataset_dict

print(f"Split size before filters: {len(raw_dataset):,}")
print(f"Columns: {raw_dataset.column_names}")

if "answerable" in raw_dataset.column_names:
    raw_dataset = raw_dataset.filter(lambda example: example["answerable"])
    print(f"After answerable filter: {len(raw_dataset):,}")

selected_size = min(MAX_QUESTIONS, len(raw_dataset))
dataset = raw_dataset.select(range(selected_size))
print(f"Selected {len(dataset)} examples")

In [ ]:
print(f"Q: {dataset[0]["question"]}")

In [ ]:
def _extract_context_pairs(example: dict) -> list[tuple[str, list[str] | str]]:
    context = example.get("context")
    pairs: list[tuple[str, list[str] | str]] = []

    if isinstance(context, dict):
        titles = context.get("title", [])
        sentences = context.get("sentences", [])
        for title, sent_list in zip(titles, sentences):
            pairs.append((str(title), sent_list))
        return pairs

    if isinstance(context, list):
        for item in context:
            if isinstance(item, dict):
                title = str(item.get("title", "")).strip()
                sent_list = item.get("sentences", item.get("text", ""))
                if title:
                    pairs.append((title, sent_list))
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                title = str(item[0]).strip()
                sent_list = item[1]
                if title:
                    pairs.append((title, sent_list))

    return pairs


def _extract_supporting_titles(example: dict) -> set[str]:
    supporting = example.get("supporting_facts")
    titles: set[str] = set()

    if isinstance(supporting, dict):
        for title in supporting.get("title", []):
            titles.add(str(title))
        return titles

    if isinstance(supporting, list):
        for item in supporting:
            if isinstance(item, dict) and "title" in item:
                titles.add(str(item["title"]))
            elif isinstance(item, (list, tuple)) and len(item) >= 1:
                titles.add(str(item[0]))

    return titles


In [ ]:
def transform_2wiki_to_loader_format(example: dict) -> dict:
    context_pairs = _extract_context_pairs(example)
    supporting_titles = _extract_supporting_titles(example)

    paragraphs = []
    for title, sentence_data in context_pairs:
        if isinstance(sentence_data, str):
            sentence_list = [sentence_data]
        else:
            sentence_list = [str(sentence) for sentence in sentence_data]

        paragraph_text = " ".join(
            sentence.strip()
            for sentence in sentence_list
            if sentence and sentence.strip()
        )
        if not paragraph_text:
            continue

        paragraphs.append({
            "title": title,
            "text": paragraph_text,
            "is_supporting": title in supporting_titles,
        })

    answer_value = example.get("answer", "")
    if isinstance(answer_value, list):
        answer = str(answer_value[0]) if answer_value else ""
    else:
        answer = str(answer_value)

    return {
        "question": str(example.get("question", "")),
        "answer": answer,
        "paragraphs": paragraphs,
    }


transformed_dataset = [
    transform_2wiki_to_loader_format(example)
    for example in dataset
]

transformed_dataset = [
    example
    for example in transformed_dataset
    if example["question"].strip() and example["answer"].strip() and example["paragraphs"]
]

print(f"Transformed {len(transformed_dataset)} examples")
print(f"Q: {transformed_dataset[0]['question']}")
print(f"A: {transformed_dataset[0]['answer']}")
print(f"Paragraphs: {len(transformed_dataset[0]['paragraphs'])}")

In [ ]:
loader = KnowledgeGraphLoader(
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
    cache_dir=Path("cache_2wiki"),
)

In [ ]:
loader.clear_and_init()

In [ ]:
ground_truth = loader.load_examples(
    transformed_dataset,
    max_extraction_workers=30,
    max_embedding_workers=5,
    save_every=1000,
)

loader.finalize()

In [ ]:
with open("ground_truth-2wiki.pkl", "wb") as f:
    pickle.dump(ground_truth, f)

print(f"Saved ground truth for {len(ground_truth)} questions")

stats = loader.get_stats()
for key, value in stats.items():
    print(f"{key}: {value}")

loader.close()